# Task 1 -- Image-to-FEN

Chessboard understanding with Qwen3.5-0.8B and token-level patch reordering (REOrder). Train on Colab; every training run and evaluation is cached on the Hugging Face Hub, so re-running a cell skips work already done.

Downloading Libraries and Imports

In [8]:
!rm -rf /usr/local/lib/python3.13/dist-packages/~orch*
!pip install -q --upgrade "pillow<11.0.0" torch torchvision transformers>=4.45.0 accelerate scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv
!pip uninstall -y torchaudio torchao -q
!pip install -q flash-linear-attention



# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
from functools import partial
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


Setting up environment

In [9]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "Ivo2",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        # /content survives "Restart session", so an old clone can linger -- pull so
        # the notebook always runs the latest code (restart the session afterwards if
        # a project module was already imported in this kernel).
        print(f"Repository directory already exists at: {repo_root}; pulling latest changes...")
        pull = subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], capture_output=True, text=True)
        print((pull.stdout or pull.stderr).strip())
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

# 4. Import custom project modules cleanly
from eval.results import (
   evaluate_with_cache,
   load_or_init_results_table,
   append_or_replace,
   load_results_table,
   push_results_table,
)

from training.checkpoints import (
   find_resumable_checkpoint,
   train_or_load_finished,
   HubBestAdapterCallback,
)

from training.preprocessing import Qwen35OnTheFlyCollator

from training.reordering import (
   get_patch_reordering_indices,
   reorder_chessboard_image,
   apply_patch_permutation,
)

from training.finetune import finetune_and_push_chessboard_model

from training.learned_reordering import train_with_learned_reordering, load_learned_order, describe_order, greedy_order
from training.finetune import experiment_repo_id, experiment_model_name
from training.reordering import resolve_square_order
from training.resolution import configure_image_resolution
from training.token_reordering import install_token_reordering, verify_token_reordering
from eval.results import evaluate_with_order
from eval.diagnostics import (
    headline_table, fen_error_breakdown, analyze_san_predictions, san_error_breakdown,
    frame_ablation_losses, training_health_report, backfill_cached_metrics, save_diagnostics,
)
import gc

from eval.diagnostics import (
    backfill_cached_metrics,
    training_health_report,
    rescore_cached_fen_results,
    save_diagnostics,
)

# Used to key every cached evaluation result on the Hub (evaluation-results-task1
# and the per-strategy LoRA repos) -- defined here, early, instead of only inside
# the "Vanilla + LoRA" section further down, since evaluate_with_cache/
# load_or_init_results_table need it as soon as the very first evaluation runs.
hf_org_prefix = "bdatm-project"

print("All custom modules and eval utilities imported successfully!")

Cloning repository from https://github.com/Aivon99/BigDataAndTextMiningProject.git...
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: NVIDIA L4
All custom modules and eval utilities imported successfully!


Downloading dataset for task 1 from HuggingFace repo

In [10]:
# Authenticate with Hugging Face:
# - Colab: reads a token from a Colab secret named HF_TOKEN (key icon, left sidebar)
# - Local: reads HF_TOKEN from a repo-root .env file, or the environment
# - Falls back to an interactive login prompt if neither is found
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment — falling back to interactive login.")
    login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task1 = load_dataset(dataset_name, num_proc=16)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

No HF_TOKEN found in secrets/.env/environment — falling back to interactive login.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

...

Setting num_proc from 16 back to 1 for the validation split to disable multiprocessing as it only contains one shard.


Generating validation split:   0%|          | 0/400 [00:00<?, ? examples/s]

Setting num_proc from 16 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': 'drw7g', 'task': 'task1', 'fen': 'r3k2r/pppq2pp/2nb1p2/1B1Q3P/8/1P2P2P/PBPP1P2/R3K2R w KQkq - 1 13', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Image: The visual representation of the chessboard.\nOutput Format:\nReturn only the valid FEN string representing the position of a

## Experiment setup

**Token-level patch reordering (REOrder).** A scan order is applied to the visual *tokens* inside the model: each image's token embeddings and their M-RoPE positions are permuted together (`src/training/token_reordering.py`), so the board image is never altered and every token keeps its true 2D position. Only the order in which the LLM reads them changes -- which matters here because 18 of Qwen3.5's 24 LLM layers are Gated DeltaNet (a recurrent scan with no positional encoding).

The first runs of this notebook rearranged *pixel tiles* instead, which changed the board itself (a "file_wise" image was a transposed board). Their results stay in the results table, labelled `legacy: pixel-tile reordering, 512px`.

**Resolution (spec 3.3, patch alignment).** At `IMAGE_SIZE = 256` the LLM sees an 8x8 token grid: exactly one token per chess square. The legacy 512px raster run is the resolution ablation (512px = 2x2 tokens per square).

In [11]:
IMAGE_SIZE = 256   # 256 -> 1 LLM token per square; 512 -> 2x2 tokens per square
TAG = f"tok{IMAGE_SIZE}"   # suffix of every Hub repo / results row of this setup
# Fixed scan orders (spec 3.3). "rank_wise" is the raster baseline itself;
# "hilbert" is one of REOrder's own orders (drop it to save a training run).
STRATEGIES = ["zigzag", "spiral_center_out", "file_wise", "hilbert"]
N_SHOT = 3   # few-shot examples for the vanilla model, drawn from train
TASK = "task1"
model_id = "Qwen/Qwen3.5-0.8B"
hf_org_prefix = "bdatm-project"

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


def load_base_model():
    return AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto"
    )


def free_gpu():
    """Call after `del`-ing the model variables, so their GPU memory is released."""
    gc.collect()
    torch.cuda.empty_cache()


for s in ["raster", *STRATEGIES]:
    print(f"{s} (reading step of each square):\n{describe_order(resolve_square_order(s))}\n")

raster (reading step of each square):
8  0  1  2  3  4  5  6  7
7  8  9 10 11 12 13 14 15
6 16 17 18 19 20 21 22 23
5 24 25 26 27 28 29 30 31
4 32 33 34 35 36 37 38 39
3 40 41 42 43 44 45 46 47
2 48 49 50 51 52 53 54 55
1 56 57 58 59 60 61 62 63
   a  b  c  d  e  f  g  h

zigzag (reading step of each square):
8  0  1  2  3  4  5  6  7
7 15 14 13 12 11 10  9  8
6 16 17 18 19 20 21 22 23
5 31 30 29 28 27 26 25 24
4 32 33 34 35 36 37 38 39
3 47 46 45 44 43 42 41 40
2 48 49 50 51 52 53 54 55
1 63 62 61 60 59 58 57 56
   a  b  c  d  e  f  g  h

spiral_center_out (reading step of each square):
8 63 62 61 60 59 58 57 56
7 36 35 34 33 32 31 30 55
6 37 16 15 14 13 12 29 54
5 38 17  4  3  2 11 28 53
4 39 18  5  0  1 10 27 52
3 40 19  6  7  8  9 26 51
2 41 20 21 22 23 24 25 50
1 42 43 44 45 46 47 48 49
   a  b  c  d  e  f  g  h

file_wise (reading step of each square):
8  0  8 16 24 32 40 48 56
7  1  9 17 25 33 41 49 57
6  2 10 18 26 34 42 50 58
5  3 11 19 27 35 43 51 59
4  4 12 20 28 36 44 52 60

## Vanilla model

The processor is set to `IMAGE_SIZE` once here; training, evaluation and generation all go through it. `verify_token_reordering` checks on this environment's `transformers` version that no-order output equals the unpatched model, that embeddings and positions move together, and that the order actually changes the output.

In [12]:
model_vanilla = load_base_model()
processor = AutoProcessor.from_pretrained(model_id)
configure_image_resolution(processor, IMAGE_SIZE)
verify_token_reordering(model_vanilla, processor, dataset_task1["test"][0]["image"], resolve_square_order("zigzag"))

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

...

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[resolution] processor set to 256px: {'image_size': 256, 'patch_grid': (16, 16), 'llm_token_grid': (8, 8), 'tokens_per_square': 1}


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[token reordering] OK: {'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.1875}


{'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.1875}

In [13]:
# Grab the first test sample
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

# Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model_vanilla.device)

# Generate the zero-shot prediction
print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=128)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Sample ID: sample_000000
FEN: 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33
Prompt provided to the model:
You are a specialized model for chessboard understanding.
Your goal is to extract the exact board state from the provided chessboard image.
Input:
- Board Image: The visual representation of the chessboard.
Output Format:
Return only the valid FEN string representing the position of all pieces on the board.

Real FEN (Ground Truth): 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33

Generating zero-shot prediction...
Predicted FEN (Zero-Shot): E2R1k1pppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppp


Zero-shot and few-shot evaluation of the vanilla model (spec, Member 1). Few-shot prepends `N_SHOT` solved train examples as earlier chat turns.

In [14]:
all_models_results = load_or_init_results_table(task=TASK, hf_org_prefix=hf_org_prefix)

vanilla_results_df, vanilla_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name("Vanilla Qwen3.5-0.8B (Zero-Shot)", TAG),
)
all_models_results = append_or_replace(all_models_results, vanilla_summary_df)

few_shot_examples = [dataset_task1["train"][i] for i in range(N_SHOT)]
fewshot_results_df, fewshot_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name(f"Vanilla Qwen3.5-0.8B ({N_SHOT}-Shot)", TAG),
    few_shot_examples=few_shot_examples,
)
all_models_results = append_or_replace(all_models_results, fewshot_summary_df)
display(fewshot_results_df.head() if fewshot_results_df is not None else fewshot_summary_df)
display(headline_table(all_models_results, TASK))
del model_vanilla
free_gpu()

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 5.26kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Loaded existing comparison table for task1 from 'bdatm-project/evaluation-results-task1' (9 model(s) already evaluated).


Evaluating Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256]: 100%|██████████| 400/400 [29:14<00:00,  4.39s/it]



Evaluation completed for Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256]! Results saved to task1_vanilla_qwen3.5-0.8b_(zero-shot)_[tok256]_results.csv.


...

                              : 100%|##########| 9.25kB / 9.25kB            

Validating                    :           |   0%            

[task1] Cached evaluation for 'Vanilla Qwen3.5-0.8B (3-Shot) [tok256]' -- pushed to 'bdatm-project/evaluation-results-task1'.


,sample_id,ground_truth,predicted,raw_output,fen_exact_match,levenshtein_distance,character_error_rate,square_by_square_accuracy,board_exact_match,board_character_error_rate,side_to_move_match,castling_match,en_passant_match,clocks_match
0,sample_000000,6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - -...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,0.0,40,0.784314,0.0,0,0.825000,0,0,1,0
1,sample_000001,r1r5/p6n/4pq2/3P1p1k/7P/5PQ1/P5P1/R3R1K1 w - -...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,0.0,40,0.784314,0.0,0,0.825000,1,0,1,0
2,sample_000002,8/6pk/5np1/1p5p/p1pP3q/P1P1Q3/1P3P2/5RK1 w - -...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1P/PPP1PPP...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1P/PPP1PPP...,0.0,35,0.686275,0.0,0,0.825000,1,1,1,0
3,sample_000003,r4r1k/6p1/p5P1/1p2ppNP/1q2b2Q/4n3/PPPRB3/1K4R1...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,0.0,41,0.719298,0.0,0,0.739130,1,0,1,0
4,sample_000004,4rrk1/6pp/p1p1q3/3p2Q1/2PP2b1/1PR2N2/P5PP/5RK1...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,0.0,37,0.649123,0.0,0,0.673913,1,0,1,0


,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],"token reordering, 256px",0.0,4.336548,0.000000,0.0000,3.412487,178.8675
1,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],"token reordering, 256px",0.0,0.750878,0.074805,0.0000,0.686740,36.0350
2,Vanilla Qwen2.5-VL (Zero-Shot),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.000000,0.0000,3.259093,169.2250
3,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.994844,0.0075,0.043934,2.2775
4,Qwen + LoRA (Zigzag - TF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.500781,0.0000,0.314656,16.4500
5,Qwen + LoRA (Spiral - TF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.379414,0.0000,0.621696,33.0075
6,Qwen + LoRA (File_wise - TF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.570391,0.0000,0.653481,34.9800
7,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.974492,0.0125,0.044541,2.2975
8,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.996680,0.0125,0.046081,2.3850
9,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.999648,0.0225,0.044630,2.3050


## LoRA baseline (raster order)

Supervised LoRA fine-tuning with the native raster order (= rank-wise). Up to 10 epochs with early stopping on validation loss; the best epoch is kept, also across a Colab disconnect.

In [15]:
raster_repo = experiment_repo_id(TASK, "raster", TAG, hf_org_prefix)
trained = finetune_and_push_chessboard_model(
    "raster", dataset_task1, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
)
del trained
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK1 | STRATEGY: RASTER (token-level) -> bdatm-project/qwen-task1-raster-tok256-lora
Training from scratch...


Epoch,Training Loss,Validation Loss
1,0.186018,0.177470
2,0.130747,0.130417
3,0.109899,0.135591
4,0.112598,0.121377
5,0.113897,0.114658
6,0.106349,0.111473
7,0.103708,0.111305
8,0.097539,0.112961
9,0.085025,0.120836


Pushing model and processor to Hugging Face Hub: bdatm-project/qwen-task1-raster-tok256-lora...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpln4i9i0r/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

Validating                    :           |   0%            

Finished! Successfully uploaded to Hub: bdatm-project/qwen-task1-raster-tok256-lora


In [16]:
raster_model = PeftModel.from_pretrained(load_base_model(), raster_repo)
raster_results_df, raster_summary_df = evaluate_with_order(
    TASK, raster_model, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix, adapter_repo_id=raster_repo,
    model_name=experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG),
)
all_models_results = append_or_replace(all_models_results, raster_summary_df)
display(headline_table(all_models_results, TASK))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.25kB            

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 9.40kB / 9.40kB            

Validating                    :           |   0%            

[task1] Cached evaluation for 'Qwen + LoRA (Raster - SFT) [tok256]' -- pushed to 'bdatm-project/evaluation-results-task1'.


,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.97,0.000838,0.984766,0.0175,0.043193,2.2375
1,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],"token reordering, 256px",0.00,4.336548,0.000000,0.0000,3.412487,178.8675
2,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],"token reordering, 256px",0.00,0.750878,0.074805,0.0000,0.686740,36.0350
3,Vanilla Qwen2.5-VL (Zero-Shot),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.000000,0.0000,3.259093,169.2250
4,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.994844,0.0075,0.043934,2.2775
5,Qwen + LoRA (Zigzag - TF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.500781,0.0000,0.314656,16.4500
6,Qwen + LoRA (Spiral - TF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.379414,0.0000,0.621696,33.0075
7,Qwen + LoRA (File_wise - TF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.570391,0.0000,0.653481,34.9800
8,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.974492,0.0125,0.044541,2.2975
9,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.996680,0.0125,0.046081,2.3850


## Training-free reordering

The raster-trained LoRA model reads the *same* images with a different token order -- no retraining. Since the board is intact, this measures how much the model depends on the order it was trained with.

In [17]:
for strat in STRATEGIES:
    _, summary = evaluate_with_order(
        TASK, raster_model, processor, dataset_task1["test"], order=strat, hf_org_prefix=hf_org_prefix,
        adapter_repo_id=raster_repo, model_name=experiment_model_name(f"Qwen + LoRA Raster, read {strat} (TF)", TAG),
    )
    all_models_results = append_or_replace(all_models_results, summary)
display(headline_table(all_models_results, TASK))

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.40kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Evaluating Qwen + LoRA Raster, read zigzag (TF) [tok256]: 100%|██████████| 400/400 [22:10<00:00,  3.33s/it]



Evaluation completed for Qwen + LoRA Raster, read zigzag (TF) [tok256]! Results saved to task1_qwen_+_lora_raster,_read_zigzag_(tf)_[tok256]_results.csv.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 9.70kB / 9.70kB            

Validating                    :           |   0%            

[task1] Cached evaluation for 'Qwen + LoRA Raster, read hilbert (TF) [tok256]' -- pushed to 'bdatm-project/evaluation-results-task1'.


,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.97,0.000838,0.984766,0.0175,0.043193,2.2375
1,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],"token reordering, 256px",0.00,4.336548,0.000000,0.0000,3.412487,178.8675
2,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],"token reordering, 256px",0.00,0.750878,0.074805,0.0000,0.686740,36.0350
3,"Qwen + LoRA Raster, read zigzag (TF) [tok256]","token reordering, 256px",0.00,0.554096,0.000000,0.0000,0.511075,27.3725
4,"Qwen + LoRA Raster, read spiral_center_out (TF...","token reordering, 256px",0.00,0.896589,0.000000,0.0000,0.786316,42.3825
5,"Qwen + LoRA Raster, read file_wise (TF) [tok256]","token reordering, 256px",0.00,0.911124,0.003516,0.0000,0.798078,42.8125
6,"Qwen + LoRA Raster, read hilbert (TF) [tok256]","token reordering, 256px",0.00,0.787052,0.006836,0.0000,0.714656,38.5125
7,Vanilla Qwen2.5-VL (Zero-Shot),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.000000,0.0000,3.259093,169.2250
8,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.994844,0.0075,0.043934,2.2775
9,Qwen + LoRA (Zigzag - TF),"legacy: pixel-tile reordering, 512px",NaN,NaN,0.500781,0.0000,0.314656,16.4500


## Fine-tuning with each scan order (SFT)

One LoRA run per strategy, trained and evaluated with that token order. Each run loads a fresh base model and frees it afterwards.

In [18]:
for strat in STRATEGIES:
    trained = finetune_and_push_chessboard_model(
        strat, dataset_task1, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    )
    del trained
    free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK1 | STRATEGY: ZIGZAG (token-level) -> bdatm-project/qwen-task1-zigzag-tok256-lora
Training from scratch...


Epoch,Training Loss,Validation Loss
1,0.248529,0.214702
2,0.169579,0.157838
3,0.120752,0.127409
4,0.117539,0.122118
5,0.117271,0.115838
6,0.111161,0.113355
7,0.108055,0.111896
8,0.101161,0.114522
9,0.092784,0.116160


Pushing model and processor to Hugging Face Hub: bdatm-project/qwen-task1-zigzag-tok256-lora...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpg7_9grut/tokenizer.json:  80%|#######9  | 16.0MB / 20.0MB            

Validating                    :           |   0%            

Finished! Successfully uploaded to Hub: bdatm-project/qwen-task1-hilbert-tok256-lora


In [19]:
for strat in STRATEGIES:
    repo = experiment_repo_id(TASK, strat, TAG, hf_org_prefix)
    strat_model = PeftModel.from_pretrained(load_base_model(), repo)
    _, summary = evaluate_with_order(
        TASK, strat_model, processor, dataset_task1["test"], order=strat, hf_org_prefix=hf_org_prefix, adapter_repo_id=repo,
        model_name=experiment_model_name(f"Qwen + LoRA ({strat.capitalize()} - SFT)", TAG),
    )
    all_models_results = append_or_replace(all_models_results, summary)
    del strat_model
    free_gpu()
display(headline_table(all_models_results, TASK))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.70kB            

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 10.4kB / 10.4kB            

Validating                    :           |   0%            

[task1] Cached evaluation for 'Qwen + LoRA (Hilbert - SFT) [tok256]' -- pushed to 'bdatm-project/evaluation-results-task1'.


,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.9850,0.000563,0.999727,0.0100,0.043812,2.2700
1,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.9725,0.000958,0.996953,0.0225,0.043943,2.2800
2,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.9700,0.000838,0.984766,0.0175,0.043193,2.2375
3,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.9675,0.002206,0.992070,0.0150,0.044137,2.2850
4,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.9550,0.001258,0.979570,0.0275,0.043633,2.2625
5,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],"token reordering, 256px",0.0000,0.750878,0.074805,0.0000,0.686740,36.0350
6,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],"token reordering, 256px",0.0000,4.336548,0.000000,0.0000,3.412487,178.8675
7,"Qwen + LoRA Raster, read hilbert (TF) [tok256]","token reordering, 256px",0.0000,0.787052,0.006836,0.0000,0.714656,38.5125
8,"Qwen + LoRA Raster, read file_wise (TF) [tok256]","token reordering, 256px",0.0000,0.911124,0.003516,0.0000,0.798078,42.8125
9,"Qwen + LoRA Raster, read spiral_center_out (TF...","token reordering, 256px",0.0000,0.896589,0.000000,0.0000,0.786316,42.3825


## Learned reordering (REOrder)

REOrder's Plackett-Luce policy (`external/REOrder`) over the 64 squares, trained jointly with LoRA by REINFORCE (`src/training/learned_reordering.py`): one warm-up epoch in raster order, then the policy samples orders with a Gumbel temperature annealed 0.2 -> 0. Reward = minus the answer's cross-entropy; two orders are scored on each sample and compared with each other, so puzzle difficulty cancels out. The best epoch by validation loss is kept, together with the order it was validated with (the policy's greedy order, or raster if the warm-up epoch was best). Resumable from Hub checkpoints.

In [ ]:
learned_model, learned_policy, learned_repo, learned_order = train_with_learned_reordering(
    dataset_task1, processor, load_base_model(), peft_config, task=TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    num_train_epochs=4, policy_start_after_epochs=1,
)
print("Order used for evaluation (reading step of each square):")
print(describe_order(learned_order))
print("\nPolicy's greedy order:")
print(describe_order(greedy_order(learned_policy)))

_, learned_summary = evaluate_with_order(
    TASK, learned_model, processor, dataset_task1["test"], order=learned_order, hf_org_prefix=hf_org_prefix,
    adapter_repo_id=learned_repo, model_name=experiment_model_name("Qwen + LoRA (Learned Reordering)", TAG),
)
all_models_results = append_or_replace(all_models_results, learned_summary)
display(headline_table(all_models_results, TASK))
del learned_model
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task1] Learned reordering: starting fresh -> bdatm-project/qwen-task1-learned-tok256-lora
[task1] epoch 0 step 50: lm_loss=1.5640
[task1] epoch 0 step 100: lm_loss=1.1463
[task1] epoch 0 step 150: lm_loss=0.6612
[task1] epoch 0 step 200: lm_loss=0.3987
[task1] epoch 0 step 250: lm_loss=0.2855
[task1] epoch 0 step 300: lm_loss=0.2157
[task1] epoch 0 step 350: lm_loss=0.2030
[task1] epoch 0 step 400: lm_loss=0.1840
[task1] epoch 0 done: val_loss=0.1673 (order: raster)
[task1] epoch 1 step 450: lm_loss=1.2968 |adv|=0.0795 temp=0.192
[task1] epoch 1 step 500: lm_loss=0.6094 |adv|=0.0537 temp=0.183
[task1] epoch 1 step 550: lm_loss=0.3700 |adv|=0.0484 temp=0.175
[task1] epoch 1 step 600: lm_loss=0.2736 |adv|=0.0419 temp=0.167
[task1] epoch 1 step 650: lm_loss=0.2110 |adv|=0.0313 temp=0.159
[task1] epoch 1 step 700: lm_loss=0.1801 |adv|=0.0242 temp=0.150
[task1] epoch 1 step 750: lm_loss=0.1806 |adv|=0.0240 temp=0.142
[task1] epoch 1 step 800: lm_loss=0.1704 |adv|=0.0235 temp=0.134
[task1] 

## Resolution ablation

Same raster LoRA recipe at 512px (legacy run: 2x2 tokens per square) vs 256px (1 token per square).

In [ ]:
LEGACY_RASTER_512 = "Qwen + LoRA Fine-Tuning (from HF)"   # raster LoRA of the first runs, 512px images
rows = all_models_results["model_name"].isin([LEGACY_RASTER_512, experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG)])
display(headline_table(all_models_results[rows], TASK))

## Results

Diagnostic columns are recomputed from the cached per-sample predictions (no inference), then the table is pushed to the Hub and reloaded from there. `headline_table` shows the spec's metrics first.

In [ ]:
all_models_results = backfill_cached_metrics(
    all_models_results, TASK, fens=None, hf_org_prefix=hf_org_prefix
)
results_repo = f"{hf_org_prefix}/evaluation-results-{TASK}"
push_results_table(results_repo, all_models_results, commit_message=f"{TASK}: comparison table")
display(headline_table(load_results_table(results_repo), TASK))

## Diagnostics

`training_health_report` per run: eval-loss history, discontinuities, recipe markers, best vs published epoch.

In [ ]:
if CONFIG["colab"]:
    from google.colab import drive
    drive.mount("/content/drive")   # save_diagnostics writes here, surviving a runtime reset

health = {r: training_health_report(r) for r in [experiment_repo_id(TASK, s, TAG, hf_org_prefix) for s in ["raster", *STRATEGIES, "learned"]]}
health_df = pd.DataFrame([
    {"repo_id": r, "n_problems": len(h["problems"]), "problems": " | ".join(h["problems"])} for r, h in health.items()
])
display(health_df)
save_diagnostics(health_df, f"training_health_{TAG}", TASK)

Qualitative error analysis of the raster LoRA model: how many squares are wrong per board, which pieces get confused, and examples.

In [ ]:
breakdown = fen_error_breakdown(raster_results_df)
display(breakdown["wrong_squares_per_board"])
display(breakdown["confusions"].head(15))
display(breakdown["examples"])
save_diagnostics(raster_results_df, f"raster_lora_per_sample_{TAG}", TASK)

In [ ]:
# Save this notebook (with outputs) to GitHub, then release the Colab runtime.
import json, subprocess
from google.colab import _message, runtime, userdata

NOTEBOOK_PATH = "notebooks/Task_1.ipynb"   # Task_2 / Task_3 in the other notebooks
BRANCH = CONFIG["branch"]
DISCONNECT = True

def git(*args):
    out = subprocess.run(["git", "-C", str(repo_root), *args], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{out.stderr}")
    return out.stdout.strip()

nb = _message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]
nb.get("metadata", {}).pop("widgets", None)   # Colab widget metadata breaks GitHub's notebook preview
with open(repo_root / NOTEBOOK_PATH, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=2)
    f.write("\n")

token = userdata.get("GITHUB_TOKEN_BIG")
remote = f"https://{token}@github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"
git("config", "user.name", "Colab run")
git("config", "user.email", "colab@users.noreply.github.com")
git("add", NOTEBOOK_PATH)
if git("status", "--porcelain", NOTEBOOK_PATH):
    git("commit", "-m", f"Run {NOTEBOOK_PATH} on Colab")
    git("pull", "--rebase", remote, BRANCH)   # pick up commits pushed since the clone
    git("push", remote, f"HEAD:{BRANCH}")
    print(f"Pushed {NOTEBOOK_PATH} to {BRANCH}: {git('log', '-1', '--format=%h %s')}")
else:
    print("Notebook unchanged -- nothing to push.")

if DISCONNECT:
    runtime.unassign()   # releases the GPU; the open tab stays readable
